In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from numpy import linalg as LA
import copy
import networkx as nx
import random

plt.rcParams.update({'font.size': 17})

import matplotlib.ticker as ticker


from scipy.stats import pearsonr



In [ ]:
#Read Supra adjacency matrix (without snap matrix)

corr_data = abs(pd.read_csv("data/insulin_eQTL_spearman_corr_matrix_with_snap_removed_CF_lt_R_latest_15k.csv"))

corr_data

In [3]:
N = int(corr_data.shape[0])
L = 2
n = int(N/L)
print(N)
print("genes per layer: ", n)

24394
genes per layer:  12197


In [4]:
# Loading Source and target genes

genes = list(corr_data.columns[:n])


# Since we have NO query set, these are ALL genes in source and target genes
source_genes = list(pd.read_csv('data/insulin_source_genes_full.csv', header=None).iloc[:,0])
target_genes = list(pd.read_csv('data/insulin_target_genes_full.csv', header=None).iloc[:,0])

print("\nSource genes  total:", len(source_genes))
print("Target genes  total:", len(target_genes))

# Find which source/target genes are present in our gene list
common_source_genes = np.intersect1d(genes, source_genes)
common_target_genes = np.intersect1d(genes, target_genes)

print("Source genes found in network:", len(common_source_genes))
print("Target genes found in network:", len(common_target_genes))



print("Statistics of the ground truth data")
print("Number of genes: ", len(genes))
print("Number of target genes: ", len(target_genes))
print("Number of source genes: ", len(source_genes))




# Get indices of these genes in the gene list
# These are used to set the query/seed vector in centrality computation
source_genes_indices = [i for i, e in enumerate(genes) 
                         if e in common_source_genes]
target_genes_indices = [i for i, e in enumerate(genes) 
                         if e in common_target_genes]

print("Source gene indices count:", len(source_genes_indices))
print("Target gene indices count:", len(target_genes_indices))


Source genes  total: 2860
Target genes  total: 6482
Source genes found in network: 106
Target genes found in network: 119
Statistics of the ground truth data
Number of genes:  12197
Number of target genes:  6482
Number of source genes:  2860
Source gene indices count: 106
Target gene indices count: 119


In [5]:
# Loading SNAP PPI data and building SNAP adjacency matrix


SNAP_data = pd.read_csv("data/SNAP_data/PPT-Ohmnet_gene_symbols.csv", index_col=0)
#SNAP_data
pancreas_df = SNAP_data.loc[SNAP_data['tissue'] == "pancreas"].values
skeletal_muscle_df = SNAP_data.loc[SNAP_data['tissue'] == "skeletal_muscle"].values


print("Pancreas PPI edges:", pancreas_df.shape[0])
print("Skeletal muscle PPI edges:", skeletal_muscle_df.shape[0])


Pancreas PPI edges: 47402
Skeletal muscle PPI edges: 46055


In [6]:

# not in use as SNAP data is not present

# Build SNAP adjacency matrix — same size as supra-adjacency
A_SNAP = np.zeros_like(corr_data.values, dtype=np.float32)

# Liver PPI edges go in the first block [0:n, 0:n]
n_offset_liver = 0
for i in range(pancreas_df.shape[0]):
    try:
        from_gene = int(genes.index(pancreas_df[i, 0])) + n_offset_liver
        to_gene   = int(genes.index(pancreas_df[i, 1])) + n_offset_liver
        A_SNAP[from_gene, to_gene] = 1
        A_SNAP[to_gene, from_gene] = 1
    except ValueError:
        pass  # gene not in our gene list, skip             
    

# Brain PPI edges go in the second block [n:2n, n:2n]
n_offset_brain = n
for i in range(skeletal_muscle_df.shape[0]):
    try:
        from_gene = int(genes.index(skeletal_muscle_df[i, 0])) + n_offset_brain
        to_gene   = int(genes.index(skeletal_muscle_df[i, 1])) + n_offset_brain
        A_SNAP[from_gene, to_gene] = 1
        A_SNAP[to_gene, from_gene] = 1
    except ValueError:
        pass

print("SNAP matrix non-zero entries:", np.count_nonzero(A_SNAP))
# Final adjacency matrix = coexpression + PPI
A_final = corr_data.values + A_SNAP


SNAP matrix non-zero entries: 11532


In [7]:
# A_final = corr_data.values  # pure coexpression network
# print("Using coexpression-only network")
print("Matrix shape:", A_final.shape)
print("Matrix range: min =", A_final.min(), "max =", A_final.max())

Matrix shape: (24394, 24394)
Matrix range: min = 3.33621248470764e-07 max = 2.0


In [8]:
# Centrality computation functions

# Basic methods: These will be used while computing the centrality vector

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))

In [9]:
# MultiCens local-set centrality

def right_new_local_centrality_st(A_tilde_full, num_layers, target_tissue, target_gene_indices, p):
    print("target tissue id")
    print(target_tissue)
    print("right_new_local_centrality")
    A_tilde_full = A_tilde_full/np.sum(A_tilde_full, axis=0)
    #A_tilde_full = A_tilde_full/LA.norm(A_tilde_full)
    num_target_genes = int(np.shape(target_gene_indices)[0])
    n = int(np.shape(A_tilde_full)[0]/num_layers)
    N = int(np.shape(A_tilde_full)[0])
    A_tilde = np.zeros_like(A_tilde_full, dtype=np.float32)
    
    if target_tissue == 1:
        A_tilde[n:,n:] = A_tilde_full[n:,n:]
    elif target_tissue == 0:
        A_tilde[:n,:n] = A_tilde_full[:n,:n]
    else:
        print("wrong target tissue")
    np.fill_diagonal(A_tilde, 0.0)
    #A_tilde = A_tilde/LA.norm(A_tilde)
    
    ones_t = np.zeros((N,))
    ones_t[(np.asarray(target_gene_indices, dtype= np.int32) + int(n*target_tissue))] = 1/len(target_gene_indices)
    l = np.copy(ones_t)
    l_new = np.copy(ones_t)
    
    count = 0
    current_angle = np.zeros(3,)
    print("local centrality computation starts")
    while(count<200):
        print(count)
        count = count + 1
        l_new = (p* ((A_tilde).dot(l))) + (1-p)*ones_t
        current_angle[0] = current_angle[1]
        current_angle[1] = current_angle[2]
        current_angle[2] = angle_between(l, l_new)
        print(current_angle)
        if(((current_angle[1] == current_angle[2])) or (current_angle[2] == 0)):
            break
        l = copy.deepcopy(l_new)
        #l = copy.deepcopy(l_new)/LA.norm(l_new)
    print("l values")
    print(l)
    #new_l = l/l.sum()# + l_tt/l_tt.sum()
    new_l = copy.deepcopy(l)
    if target_tissue == 0:
        print("Found local centrality for target set centrality")
        new_l[:n] = l[:n]/l[:n].sum()
    elif target_tissue == 1:
        new_l[n:] = l[n:]/l[n:].sum()
        print("Found local centrality for source set centrality")
    else:
        print("invalid target tissue")
    print("Target genes")
    print(target_gene_indices)
    return new_l

In [10]:
# MultiCens: Query-set centrality
def right_target_global_centrality_t(A_tilde_full, num_layers, target_tissue, target_gene_indices, p):
  
  # Query-set centrality — main MultiCens measure
    l = right_new_local_centrality_st(A_tilde_full, num_layers, target_tissue, target_gene_indices, p) #last checkpoint
    #A_tilde = copy.deepcopy(A_tilde_full)
    A_tilde = A_tilde_full/np.sum(A_tilde_full, axis=0)
    N = int(np.shape(A_tilde)[0])
    n = int(N/num_layers)
    print("A_tilde sums")
    print(np.sum(A_tilde[0,:]))
    print(np.sum(A_tilde[:,0]))
        
    A = np.zeros_like(A_tilde_full, dtype=np.float32)
    C = np.zeros_like(A_tilde_full, dtype=np.float32)
    A[:n,:n] = copy.deepcopy(A_tilde[:n,:n])
    A[n:,n:] = copy.deepcopy(A_tilde[n:,n:])
    C[:n,n:] = copy.deepcopy(A_tilde[:n,n:])
    C[n:,:n] = copy.deepcopy(A_tilde[n:,:n])
    
    
    print("finding source global centrality for layer " + str(target_tissue))
    
    ones_t = np.zeros((N,))
    ones_t[(np.asarray(target_gene_indices, dtype= np.int32) + int(n*target_tissue))] = 1/len(target_gene_indices)
    g = copy.deepcopy(ones_t)
    g_new = copy.deepcopy(ones_t)
    
    
    counter = 0
    current_angle = np.zeros(3,)
    while(counter<200):
        g_new = (p * ((A+C).dot(g) + (C.dot(l)) )) + ((1-p)*ones_t)

        current_angle[0] = current_angle[1]
        current_angle[1] = current_angle[2]
        current_angle[2] = angle_between(g, g_new)
        print(current_angle)
        if(((current_angle[1] == current_angle[2])) or (current_angle[2] == 0)):
            break
        g = copy.deepcopy(g_new)
        #g = copy.deepcopy(g_new)/LA.norm(g_new)
        counter+=1
        print(counter)
        print(np.sum(g))
    new_g = copy.deepcopy(g)
    if target_tissue == 1:
        print("Found target set centrality")
        new_g[:n] = g[:n]/g[:n].max()
    elif target_tissue == 0:
        new_g[n:] = g[n:]/g[n:].max()
        print("Found source set centrality")
    else:
        print("invalid target tissue")
    return l, new_g


In [11]:
# MultiCens: local centrality
def local_centrality(A_tilde_full, num_layers, target_tissue, target_gene_indices, p):
    print("target tissue id")
    print(target_tissue)
    print("local_centrality")
    A_tilde_full = A_tilde_full/np.sum(A_tilde_full, axis=0)
    #A_tilde_full = A_tilde_full/LA.norm(A_tilde_full)
    num_target_genes = int(np.shape(target_gene_indices)[0])
    n = int(np.shape(A_tilde_full)[0]/num_layers)
    N = int(np.shape(A_tilde_full)[0])
    A_tilde = np.zeros_like(A_tilde_full, dtype=np.float32)
    
    for i in range(num_layers):
        A_tilde[(i*n):((i+1)*n),(i*n):((i+1)*n)] = A_tilde_full[(i*n):((i+1)*n),(i*n):((i+1)*n)]
        
    ones_t = np.ones((N,))/N
    l = np.copy(ones_t)
    l_new = np.copy(ones_t)
    
    count = 0
    current_angle = np.zeros(3,)
    print("local centrality computation starts")
    while(count<200):
        print(count)
        count = count + 1
        l_new = (p* ((A_tilde).dot(l))) + (1-p)*ones_t
        current_angle[0] = current_angle[1]
        current_angle[1] = current_angle[2]
        current_angle[2] = angle_between(l, l_new)
        print(current_angle)
        if(((current_angle[1] == current_angle[2])) or (current_angle[2] == 0)):
            break
        l = copy.deepcopy(l_new)
        #l = copy.deepcopy(l_new)/LA.norm(l_new)
    print("l values")
    print(l)
    #new_l = l/l.sum()# + l_tt/l_tt.sum()
    new_l = copy.deepcopy(l)
    
    for i in range(num_layers):
        l[(i*n):((i+1)*n)] = l[(i*n):((i+1)*n)]/l[(i*n):((i+1)*n)].sum()
    return new_l


# MultiCens: global centrality

def global_centrality(A_tilde_full, num_layers, target_tissue, target_gene_indices, p):
    l = local_centrality(A_tilde_full, num_layers, target_tissue, target_gene_indices, p) #last checkpoint
    #A_tilde = copy.deepcopy(A_tilde_full)
    A_tilde = A_tilde_full/np.sum(A_tilde_full, axis=0)
    N = int(np.shape(A_tilde)[0])
    n = int(N/num_layers)
    print("A_tilde sums")
    print(np.sum(A_tilde[0,:]))
    print(np.sum(A_tilde[:,0]))
        
    A = np.zeros_like(A_tilde_full, dtype=np.float32)
    C = np.zeros_like(A_tilde_full, dtype=np.float32)
    A[:n,:n] = copy.deepcopy(A_tilde[:n,:n])
    A[n:(2*n),n:(2*n)] = copy.deepcopy(A_tilde[n:(2*n),n:(2*n)])
    A[(2*n):(3*n),(2*n):(3*n)] = copy.deepcopy(A_tilde[(2*n):(3*n),(2*n):(3*n)])
    A[(3*n):(4*n),(3*n):(4*n)] = copy.deepcopy(A_tilde[(3*n):(4*n),(3*n):(4*n)])
    C = A_tilde - A
    
    
    print("finding global centrality ")
    
    ones_t = np.ones((N,))/N
    g = copy.deepcopy(ones_t)
    g_new = copy.deepcopy(ones_t)
    
    
    counter = 0
    current_angle = np.zeros(3,)
    
    while(counter<150):
        g_new = (p * ((A+C).dot(g) + (C.dot(l)) )) + ((1-p)*ones_t)

        current_angle[0] = current_angle[1]
        current_angle[1] = current_angle[2]
        current_angle[2] = angle_between(g, g_new)
        print(current_angle)
        if(((current_angle[1] == current_angle[2])) or (current_angle[2] == 0)):
            break
        g = copy.deepcopy(g_new)
        #g = copy.deepcopy(g_new)/LA.norm(g_new)
        counter+=1
        print(counter)
        print(np.sum(g))
    
    new_g = copy.deepcopy(g)
    g_fresh = copy.deepcopy(g)
    new_g = new_g/new_g.sum()
    g_fresh = g_fresh/g_fresh.sum()
    return l, new_g


# MultiCens: target-layer centrality

def target_layer_centrality(A_tilde_full, num_layers, target_tissue, target_gene_indices, p):
    l = local_centrality(A_tilde_full, num_layers, target_tissue, target_gene_indices, p) #last checkpoint
    #A_tilde = copy.deepcopy(A_tilde_full)
    A_tilde = A_tilde_full/np.sum(A_tilde_full, axis=0)
    N = int(np.shape(A_tilde)[0])
    n = int(N/num_layers)
    print("A_tilde sums")
    print(np.sum(A_tilde[0,:]))
    print(np.sum(A_tilde[:,0]))
        
    A = np.zeros_like(A_tilde_full, dtype=np.float32)
    C = np.zeros_like(A_tilde_full, dtype=np.float32)
    A[:n,:n] = copy.deepcopy(A_tilde[:n,:n])
    A[n:,n:] = copy.deepcopy(A_tilde[n:,n:])
    C[:n,n:] = copy.deepcopy(A_tilde[:n,n:])
    C[n:,:n] = copy.deepcopy(A_tilde[n:,:n])
    
    
    print("finding target layer centrality for layer " + str(target_tissue))
    
    ones_t = np.zeros((N,))
    ones_t[int(n*target_tissue):] = 1/N
    g = copy.deepcopy(ones_t)
    g_new = copy.deepcopy(ones_t)
    
    
    counter = 0
    current_angle = np.zeros(3,)
    while(counter<150):
        g_new = (p * ((A+C).dot(g) + (C.dot(l)) )) + ((1-p)*ones_t)

        current_angle[0] = current_angle[1]
        current_angle[1] = current_angle[2]
        current_angle[2] = angle_between(g, g_new)
        print(current_angle)
        if(((current_angle[0] == current_angle[1]) and (current_angle[1] == current_angle[2])) or (current_angle[2] == 0)):
            break
        g = copy.deepcopy(g_new)
        #g = copy.deepcopy(g_new)/LA.norm(g_new)
        counter+=1
        print(counter)
        print(np.sum(g))
    new_g = copy.deepcopy(g)
    if target_tissue == 1:
        print("Found target set centrality")
        new_g[:n] = g[:n]/g[:n].max()
    elif target_tissue == 0:
        new_g[n:] = g[n:]/g[n:].max()
        print("Found source set centrality")
    else:
        print("invalid target tissue")
    return new_g

In [ ]:
# STEP 5: Run centrality computations
# target_tissue = 1 means skeletal_muscle is the target
# → ranks PANCREAS genes by their influence ON skeletal_muscle
# target_tissue = 0 means pancreas is the target
# → ranks skeletal_muscle genes by their influence ON PANCREAS

print("\nComputing Query-set centrality (pancreas → skeletal_muscle)...")

# Rank PANCREAS genes by influence on skeletal_muscle
# target_tissue=1 → skeletal_muscle is the query/target layer
l_pancreas, g_pancreas = right_target_global_centrality_t(
    A_final,
    num_layers=2,
    target_tissue=1,           # skeletal_muscle = layer 1
    target_gene_indices=target_genes_indices,  # skeletal_muscle query genes
    p=0.9
)

print("\nComputing Local and Global centrality...")
local_cen_vector, global_cen_vector = global_centrality(
    A_final,
    num_layers=2,
    target_tissue=1,
    target_gene_indices=target_genes_indices,
    p=0.9
)

In [17]:
# STEP 6: Extract and save gene rankings
output_dir = "E:/circadian_rhythm/MultiCens/checking_output/"

# ── Pancreas gene rankings (influence on skeletal_muscle) ──────────────────
# g_pancreas[:n] contains scores for pancreas genes
pancreas_order = g_pancreas[:n].argsort()
pancreas_ranks = pancreas_order.argsort()

pancreas_results = pd.DataFrame({
    'gene_name':   genes,
    'QC_score':    g_pancreas[:n],          # query-set centrality
    'LC_score':    local_cen_vector[:n], # local centrality
    'GC_score':    global_cen_vector[:n],# global centrality
    'rank':        n - pancreas_ranks
})
pancreas_results = pancreas_results.sort_values(by='QC_score', ascending=False)

print("\n=== TOP 20 PANCREAS GENES influencing SKELETAL_MUSCLE ===")
print(pancreas_results[['gene_name', 'QC_score', 'rank']].head(20).to_string())

pancreas_results.to_csv(
    output_dir + "pancreas_genes_ranked_by_skeletal_muscle_influence.csv",
    index=False
)



=== TOP 20 PANCREAS GENES influencing SKELETAL_MUSCLE ===
      gene_name  QC_score  rank
11863  HSP90AA1  1.000000     1
5312   GAS6-AS1  0.997123     2
7955       FLT4  0.994058     3
8278      BRCA1  0.985992     4
7165      HDAC1  0.983197     5
10893     SMAD3  0.982609     6
8187        JUN  0.982188     7
11165    MPV17L  0.981371     8
10720  C1orf132  0.980684     9
10331   BHLHA15  0.979903    10
9005      CCNB1  0.979889    11
7972      ZNRF3  0.979662    12
11831      QPRT  0.979632    13
11047     KIFC3  0.979052    14
1309      ACTA1  0.978885    15
5545      FABP5  0.978703    16
10019     ESRP2  0.978531    17
11976       CFB  0.977437    18
10105      CDH1  0.977349    19
8078      MYLIP  0.977119    20


In [ ]:
# For target genes, we want to rank skeletal_muscle genes by their influence on pancreas


A_swapped = np.zeros_like(A_final)
A_swapped[:n,:n] = A_final[n:,n:]   # brain-brain block moves to layer 0
A_swapped[:n,n:] = A_final[n:,:n]
A_swapped[n:,:n] = A_final[:n,n:]
A_swapped[n:,n:] = A_final[:n,:n]   # liver-liver block moves to layer 1

# Now layer 0 = brain, layer 1 = liver
# Use liver genes as query in layer 1
l_brain, g_brain = right_target_global_centrality_t(
    A_swapped, num_layers=2,
    target_tissue=1,
    target_gene_indices=source_genes_indices,  # liver genes as query
    p=0.9
)


print("\nComputing Local and Global centrality...")

local_cen_vector_brain, global_cen_vector_brain = global_centrality(
    A_swapped, num_layers=2,
    target_tissue=1,
    target_gene_indices=source_genes_indices,  # liver genes as query
    p=0.9
)



In [ ]:
#  g_brain[:n] = brain gene scores, properly normalized
brain_order = g_brain[:n].argsort()
brain_ranks = brain_order.argsort()
brain_results = pd.DataFrame({
    'gene_name': genes,
    'QC_score':  g_brain[:n],
    'LC_score':  local_cen_vector_brain[:n],   # brain local centrality
    'GC_score':  global_cen_vector_brain[:n],  # brain global centrality
    'rank':      n - brain_ranks
})
brain_results = brain_results.sort_values(by='QC_score', ascending=False)


print("\n=== TOP 20 LIVER GENES influencing BRAIN ===")
print(brain_results[['gene_name', 'QC_score', 'rank']].head(20).to_string())

brain_results.to_csv(
    output_dir + "brain_genes_ranked_by_liver_influence.csv",
    index=False
)


=== TOP 20 LIVER GENES influencing BRAIN ===
           gene_name  QC_score  rank
6324   RP11-736K20.5  1.000000     1
3396           KRT14  0.998475     2
10085            DSP  0.996909     3
4016            SBSN  0.996675     4
8867          MRPL49  0.996102     5
11931           TUBB  0.994905     6
7593            AKT1  0.992159     7
8753         SNRPGP4  0.991619     8
7760   RP11-373D23.3  0.989655     9
9743            FZD4  0.988480    10
2015      AC008592.3  0.987565    11
10403         SPINT2  0.987351    12
2266            KRT5  0.986797    13
6792            CDK1  0.986172    14
10980       C6orf141  0.986113    15
6639    RP11-81A22.4  0.985623    16
1165         COL17A1  0.984924    17
5704           CPT1A  0.984307    18
9720            SDC1  0.983651    19
5664          NCCRP1  0.983103    20


'brain_results.to_csv(\n    output_dir + "brain_genes_ranked_by_liver_influence.csv",\n    index=False\n)'